<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PretrainedPytorchDogvsCat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import torch
import torch.nn as nn

from torchvision import datasets,transforms
from torch.utils.data import DataLoader,Subset
import warnings
import torch.optim as optim
import os
from google.colab import userdata
import timm
import pandas as pd
import numpy as np
from PIL import Image,ImageFile
from sklearn.model_selection import train_test_split

In [72]:
warnings.filterwarnings("ignore", category=ResourceWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('key and username activated!')

key and username activated!


In [4]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset

Dataset URL: https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset
License(s): other
100% 788M/788M [00:37<00:00, 22.0MB/s]



In [5]:
!unzip -q microsoft-catsvsdogs-dataset.zip -d ./dataser_folder

In [11]:
train_dir = '/content/dataser_folder/PetImages'
cat = '/content/dataser_folder/PetImages/Cat'

In [12]:
sizes = []

# Scan the first 50 images for a quick check
for file in os.listdir(cat)[:50]:
    if file.endswith(('.jpg', '.png', '.jpeg')):
        img = Image.open(os.path.join(cat, file))
        sizes.append(img.size) # (width, height)

# Analyze the results
min_size = min(sizes)
max_size = max(sizes)
avg_size = (sum(w for w, h in sizes)//len(sizes), sum(h for w, h in sizes)//len(sizes))

print(f"📏 Smallest Image: {min_size}")
print(f"📏 Largest Image: {max_size}")
print(f"📏 Average Size: {avg_size}")

📏 Smallest Image: (121, 124)
📏 Largest Image: (500, 489)
📏 Average Size: (405, 378)


In [19]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [56]:
# 1. This is the secret: Turn the Warning into a Crash so 'except' catches it
warnings.filterwarnings("error", category=UserWarning)
count = 0
for root, dirs, files in os.walk(train_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.load()  # <--- This is the "Truth" test. It forces a full read.
            except Exception as e:
                print(f"❌ Deleting incomplete/corrupt image: {file_path}")
                os.remove(file_path)
                count += 1
# 2. IMPORTANT: Turn warnings back to normal after cleaning
warnings.resetwarnings()
print(f"🧹 Cleanup finished. Total deleted: {count}")

❌ Deleting incomplete/corrupt image: /content/dataser_folder/PetImages/Dog/9041.jpg
🧹 Cleanup finished. Total deleted: 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [57]:
train_data = datasets.ImageFolder(root=train_dir,transform=train_transform)
val_data = datasets.ImageFolder(root=train_dir,transform=val_transform)

In [58]:
indics = list(range(len(train_data)))

In [59]:
train_idx,val_idx = train_test_split(indics,test_size=0.3,random_state=42,stratify=train_data.targets)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [60]:
final_train_data = Subset(train_data,train_idx)
final_val_data = Subset(val_data,val_idx)

In [61]:
train_loader = DataLoader(dataset=final_train_data,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
val_loader = DataLoader(dataset=final_val_data,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)

In [62]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [63]:
model = timm.create_model(model_name='tf_efficientnetv2_s',pretrained=True,num_classes=0)

In [64]:
# 1. Get the config from your model (run this once)
config = model.default_cfg
print(config)

{'url': 'https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-effv2-weights/tf_efficientnetv2_s_21ft1k-d7dafa41.pth', 'hf_hub_id': 'timm/tf_efficientnetv2_s.in21k_ft_in1k', 'architecture': 'tf_efficientnetv2_s', 'tag': 'in21k_ft_in1k', 'custom_load': False, 'input_size': (3, 300, 300), 'test_input_size': (3, 384, 384), 'fixed_input_size': False, 'interpolation': 'bicubic', 'crop_pct': 1.0, 'crop_mode': 'center', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'num_classes': 1000, 'pool_size': (10, 10), 'first_conv': 'conv_stem', 'classifier': 'classifier', 'license': 'apache-2.0'}


In [65]:
# First we create our own custom head
custom_head = nn.Sequential(
    nn.LazyLinear(128),
    nn.ReLU(),
    nn.Linear(128,1)
)

In [66]:
#Now we freeze the pretrained model
for param in model.parameters():
  param.requires_grad = False

#now we go into the blocks folder and pick only the last 2 sub folders
for sub_folder in list(model.blocks)[-2:]:
  for param in sub_folder.parameters():
    param.requires_grad = True

#Now unfreeze our custom head
for param in custom_head.parameters():
  param.requires_grad = True

In [67]:
final_model = nn.Sequential(
    model,
    custom_head
).to(device)

In [68]:
optimizer = optim.Adam(params=final_model.parameters(),lr=0.0001)
loss_fn = nn.BCEWithLogitsLoss()

In [69]:
tempImage,tempLabel = next(iter(train_loader))

final_model.eval()
with torch.no_grad():
  testout = final_model(tempImage.to(device))

print(testout.shape)
print(tempLabel.shape)

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=2457) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered

torch.Size([32, 1])
torch.Size([32])


In [71]:
epochs = 10

for epoch in range(epochs):
  model.train()

  train_loss = 0
  train_correct= 0
  train_total=0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    output = final_model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = (torch.sigmoid(output) > 0.5).int()
    train_correct += (pred.squeeze() == labels.squeeze()).sum().item()
    train_total += labels.size(0)
  train_accuracy = train_correct / train_total

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for valimages,vallabels in val_loader:
      valimages = valimages.to(device)
      vallabels = vallabels.float().unsqueeze(1).to(device)

      output = final_model(valimages)
      loss = loss_fn(output,vallabels)
      val_loss += loss.item()
      pred = (torch.sigmoid(output) > 0.5).int()
      val_correct += (pred.squeeze() == vallabels.squeeze()).sum().item()
      val_total += vallabels.size(0)
    val_accuracy = val_correct / val_total

    train_losses = train_loss / len(train_loader)
    val_losses = val_loss / len(val_loader)

    print(f'\nEpochs {epoch+1}/{epochs}...')
    print(f'train_accuracy: {train_accuracy} | train_loss:{train_losses}')
    print(f'val_accuracy: {val_accuracy} | val_loss:{val_losses}')

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.So


Epochs 1/10...
train_accuracy: 0.9971995199177002 | train_loss:0.008250689864244506
val_accuracy: 0.994 | val_loss:0.019132005414421463


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.So


Epochs 2/10...
train_accuracy: 0.9977138938103675 | train_loss:0.006993544870784473
val_accuracy: 0.9946666666666667 | val_loss:0.01423881288622571


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.So


Epochs 3/10...
train_accuracy: 0.9987426415957021 | train_loss:0.004831895641288679
val_accuracy: 0.9946666666666667 | val_loss:0.015098649566215636


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9f0b8a69e0>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d9dd49332a0>
  self.pid = os.fork()


KeyboardInterrupt: 